# 📝 การบ้าน Day 3: Evaluation & Optimization
## Agentic RAG: From Zero to Hero

**กำหนดส่ง:** 1 สัปดาห์

## 🎓 กรอกข้อมูลนักศึกษา

In [ ]:
# ─── กรอกข้อมูลของคุณ ───
STUDENT_NAME = ''   # เช่น 'สมชาย ใจดี'
STUDENT_ID   = ''   # เช่น '6512345678'
PHONE        = ''   # เช่น '081-234-5678'
LINE_ID      = ''   # เช่น 'somchai.j'

# ─── ตรวจสอบ (ห้ามแก้ไข) ───
assert len(STUDENT_ID) >= 5, '❌ กรุณากรอกรหัสนักศึกษา!'
assert len(STUDENT_NAME) >= 2, '❌ กรุณากรอกชื่อ-นามสกุล!'

print(f'✅ ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'✅ รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

---
## 📦 ติดตั้ง Dependencies

In [ ]:
%%time
!pip install -q ragas datasets google-adk google-genai \
    sentence-transformers qdrant-client langchain-text-splitters rank_bm25

import os, json, hashlib, random, asyncio, warnings
import numpy as np
warnings.filterwarnings('ignore')

try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
except:
    os.environ['GOOGLE_API_KEY'] = input('🔑 API Key: ')
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'False'

from google import genai
client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
print('✅ Setup complete')

## 📄 สร้างชุดข้อมูลเฉพาะตัว (ห้ามแก้ไข)

In [ ]:
%%time
# ─── Anti-Cheat: สร้างข้อมูลเฉพาะรหัสนักศึกษา ───
seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (10**9)
rng = random.Random(seed)

all_topics = {
    'healthcare': [
        'โรงพยาบาลศิริราชใช้ AI วิเคราะห์ภาพ X-ray ทรวงอก แม่นยำ 95% ลดเวลาวินิจฉัยจาก 30 นาทีเหลือ 5 นาที',
        'NLP วิเคราะห์เวชระเบียนอิเล็กทรอนิกส์ช่วยแพทย์สรุปประวัติผู้ป่วยได้เร็วขึ้น ลดเวลาจาก 15 นาทีเหลือ 2 นาที',
        'ระบบ AI ช่วยคัดกรองเบาหวานจากภาพถ่ายจอประสาทตา ใช้ Deep Learning ตรวจพบได้ตั้งแต่ระยะเริ่มต้น',
    ],
    'banking': [
        'ธนาคารกสิกรไทยใช้ RAG ตอบคำถามลูกค้า ลดภาระ Call Center 40% สามารถบริการ 24 ชั่วโมง',
        'ระบบตรวจจับการฉ้อโกงใช้ Machine Learning วิเคราะห์ธุรกรรมแบบ real-time ลดการฉ้อโกง 60%',
        'AI วิเคราะห์ความเสี่ยงสินเชื่อจากข้อมูลทางเลือก เช่น ประวัติค่าโทรศัพท์ ค่าน้ำค่าไฟ',
    ],
    'education': [
        'จุฬาลงกรณ์สร้างระบบ RAG ถาม-ตอบจากตำรา 500 เล่ม นักศึกษาเรียนรู้ด้วยตนเอง 24 ชั่วโมง',
        'Intelligent Tutoring System ปรับเนื้อหาตามระดับผู้เรียน ใช้ adaptive learning algorithm',
        'AI ช่วยตรวจข้อสอบอัตนัยภาษาไทย ลดเวลาตรวจ 70% ความแม่นยำ 88% เมื่อเทียบกับอาจารย์',
    ],
    'agriculture': [
        'Smart Farming ใช้ AI วิเคราะห์ภาพจากโดรน ตรวจโรคพืช 8 ชนิด ความแม่นยำ 92%',
        'ระบบพยากรณ์ผลผลิตข้าวจากข้อมูลดาวเทียม IoT sensor และสภาพอากาศ แม่นยำ 85%',
        'AI วิเคราะห์ราคาสินค้าเกษตรจากข่าวสารและ Social Media ช่วยเกษตรกรตัดสินใจการขาย',
    ],
    'logistics': [
        'ระบบ AI เพิ่มประสิทธิภาพเส้นทางขนส่งลดต้นทุนน้ำมัน 25% ใช้ Graph Neural Network',
        'คลังสินค้าอัจฉริยะใช้หุ่นยนต์ AGV และ Computer Vision จัดเรียงสินค้าอัตโนมัติ',
        'Chatbot บริการลูกค้าขนส่งใช้ NLP ภาษาไทย ตอบสถานะพัสดุและราคาส่งแบบ real-time',
    ],
}

topics = rng.sample(list(all_topics.keys()), 4)
my_data = {}
for t in topics:
    my_data[t] = rng.sample(all_topics[t], 2)

print(f'📋 Topics ของคุณ ({STUDENT_ID}):')
for t, texts in my_data.items():
    print(f'  📌 {t}: {len(texts)} texts')
    for tx in texts:
        print(f'     → {tx[:50]}...')

---
## 🎯 ขั้นตอนที่ 1: สร้าง RAG Pipeline (3 คะแนน)

สร้าง pipeline จากข้อมูล `my_data`:
1. Chunk ด้วย RecursiveCharacterTextSplitter
2. Embed ด้วย multilingual-e5-large
3. Upsert เข้า Qdrant
4. สร้างฟังก์ชัน `search_qdrant(query)` + `rag_answer(question)`

In [ ]:
%%time
# ─── ขั้นตอนที่ 1: สร้าง RAG Pipeline ───
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 💡 Hint:
#   1. embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
#   2. splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
#   3. วนลูป my_data → chunk → embed → เก็บใน list
#   4. qdrant = QdrantClient(':memory:') → create_collection → upsert
#   5. สร้าง search_qdrant(query, top_k=3) → return [{text, source, score}]
#   6. สร้าง rag_answer(question) → search + Gemini → return answer

# embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
# splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
#
# all_chunks, all_sources = [], []
# for src, texts in my_data.items():
#     for text in texts:
#         for chunk in splitter.split_text(text):
#             all_chunks.append(chunk)
#             all_sources.append(src)
#
# def search_qdrant(query, top_k=3):
#     ...
#
# def rag_answer(question, top_k=3):
#     ...


---
## 🎯 ขั้นตอนที่ 2: วัดคุณภาพ RAG (2.5 คะแนน)

1. สร้าง eval questions อย่างน้อย 5 ข้อ (จากข้อมูลของตัวเอง)
2. วัดด้วย **LLM-as-Judge** ให้คะแนน 1-5
3. แสดงผลคะแนนเฉลี่ย

In [ ]:
%%time
# ─── ขั้นตอนที่ 2: วัดคุณภาพ ───

# 💡 Hint:
#   1. สร้าง eval_questions = ['คำถาม1', 'คำถาม2', ...] อย่างน้อย 5 ข้อ
#   2. วน rag_answer() เก็บ answers
#   3. ใช้ llm_judge() ให้คะแนนแต่ละข้อ
#   4. คำนวณค่าเฉลี่ย

def llm_judge(question, answer):
    prompt = f'''ให้คะแนน 1-5:
Q: {question}
A: {answer}
ตอบ JSON: {{"score": 1-5, "reason": "..."}}'''
    resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json'))
    return json.loads(resp.text)

# eval_questions = [
#     '...',
#     '...',
# ]
# scores = []
# for q in eval_questions:
#     ans, _ = rag_answer(q)
#     verdict = llm_judge(q, ans)
#     scores.append(verdict['score'])
#     print(f"  {'⭐'*verdict['score']} {q[:30]}...")
# print(f'Average: {sum(scores)/len(scores):.1f}/5.0')


---
## 🎯 ขั้นตอนที่ 3: ปรับปรุง Pipeline (2.5 คะแนน)

1. ปรับปรุงอย่างน้อย **2 อย่าง** (เช่น chunk_size + prompt)
2. วัดผล **Before/After** ด้วย LLM-as-Judge
3. อธิบายว่า **ทำไม** ดีขึ้นหรือไม่ดีขึ้น

In [ ]:
%%time
# ─── ขั้นตอนที่ 3: ปรับปรุง ───

# 💡 Hint:
#   1. บันทึก baseline_avg จากขั้นตอนที่ 2
#   2. ปรับ chunk_size เช่น 100→300 → re-embed → re-search → วัดอีกครั้ง
#   3. ปรับ prompt ใน rag_answer เช่น เพิ่ม "ตอบกระชับ อ้างอิงข้อมูล"
#   4. เปรียบเทียบ before vs after

# baseline_avg = ...  # จากขั้นตอนที่ 2
# 
# # ปรับปรุง 1: เปลี่ยน chunk_size
# # ปรับปรุง 2: เปลี่ยน prompt
# 
# improved_avg = ...
# print(f'Before: {baseline_avg:.1f} → After: {improved_avg:.1f}')
# print(f'เหตุผล: ...')


---
## 🎯 ขั้นตอนที่ 4: สร้าง Agent + Test (2 คะแนน)

1. สร้าง Agent ด้วย Google ADK ที่ใช้ Tool ค้นหาจาก RAG
2. เขียน test cases อย่างน้อย **5 ข้อ**
3. วัด pass rate

In [ ]:
%%time
# ─── ขั้นตอนที่ 4: Agent + Testing ───
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# 💡 Hint:
#   1. สร้าง tool ที่เรียก search_qdrant()
#      def search_knowledge(query: str) -> dict:
#          results = search_qdrant(query, top_k=3)
#          return {'results': results}
#
#   2. สร้าง Agent
#      my_agent = Agent(name='hw3', model='gemini-2.5-flash',
#          instruction='...', tools=[search_knowledge])
#
#   3. เขียน test cases
#      test_cases = [
#          {'input': '...', 'expected_tool': 'search_knowledge'},
#          {'input': '...', 'expected_tool': None},
#      ]
#
#   4. วน test แล้ววัด pass rate


---
## 📊 เกณฑ์การให้คะแนน

| ขั้นตอน | คะแนน | เกณฑ์ |
|---------|:-----:|------|
| 1. RAG Pipeline | 3 | Pipeline ทำงาน, ค้นหาได้, rag_answer ตอบได้ |
| 2. วัดคุณภาพ | 2.5 | eval questions ≥5, มีคะแนน LLM-as-Judge |
| 3. ปรับปรุง | 2.5 | ปรับ ≥2 อย่าง, มี Before/After, อธิบายเหตุผล |
| 4. Agent + Test | 2 | Agent ทำงาน, test cases ≥5, มี pass rate |
| **รวม** | **10** | |

## ✅ ตรวจสอบคำตอบ

In [ ]:
# ─── ตรวจสอบก่อนส่ง (ห้ามแก้ไข) ───
print('═' * 50)
print(f'📋 สรุปการบ้าน Day 3')
print(f'═' * 50)
print(f'ชื่อ: {STUDENT_NAME}')
print(f'รหัส: {STUDENT_ID}')
print(f'Topics: {list(my_data.keys())}')

checks = {
    'RAG Pipeline': 'search_qdrant' in dir() or 'search_qdrant' in globals(),
    'rag_answer': 'rag_answer' in dir() or 'rag_answer' in globals(),
    'llm_judge': 'llm_judge' in dir() or 'llm_judge' in globals(),
}

for name, ok in checks.items():
    print(f"  {'✅' if ok else '❌'} {name}")

if all(checks.values()):
    print('\n🎉 พร้อมส่ง!')
else:
    print('\n⚠️ ยังไม่ครบ กรุณาทำส่วนที่ขาด')